# TCGA-LGG nnU-Net 2D Training — Single Fold

Run this notebook on Kaggle (GPU T4 x2 or P100).  
**Before running:** set `FOLD` in the config cell to your assigned fold.

| Person | FOLD |
|--------|------|
| Person A | 0 |
| Person B | 1 |
| Person C | 2 |

**Kaggle setup:**
1. Add the `mateuszbuda/lgg-mri-segmentation` dataset as input (Notebook → Add data → Search Kaggle datasets).
2. Set your assigned `FOLD` below.
3. Run All.

Expected runtime: ~2–6 hours per fold.

In [ ]:
# ── CONFIG — edit only this cell ─────────────────────────────────────────
FOLD = 0  # your assigned fold: 0, 1, or 2

# Path to the LGG dataset added as Kaggle input
# (after adding mateuszbuda/lgg-mri-segmentation, it will be at this path)
LGG_INPUT = "/kaggle/input/lgg-mri-segmentation"
# ─────────────────────────────────────────────────────────────────────────

## 1. Install dependencies

In [ ]:
import subprocess, sys

def run(cmd, check=True):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
    return result

# Kaggle already has PyTorch + CUDA; just add nnunetv2 and Pillow
run("pip install nnunetv2 Pillow --quiet")

## 2. Set up nnU-Net environment variables

In [ ]:
import os
from pathlib import Path

NNUNET_RAW          = Path("/kaggle/working/nnUNet_raw")
NNUNET_PREPROCESSED = Path("/kaggle/working/nnUNet_preprocessed")
NNUNET_RESULTS      = Path("/kaggle/working/nnUNet_results")

for d in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

os.environ["nnUNet_raw"]          = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"]      = str(NNUNET_RESULTS)

print("nnUNet_raw         :", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results     :", os.environ["nnUNet_results"])

## 3. Convert TCGA-LGG TIFF slices to nnU-Net 2D format

Each 2D FLAIR slice becomes a separate nnU-Net case (`{patient_id}_{slice_num:03d}`).  
All ~3929 slices (tumor + non-tumor) are included; nnU-Net handles class imbalance internally.

In [ ]:
# Copy the conversion script from the repo (edit the path if needed)
# If running standalone, paste the script content directly or upload it as a dataset.
import re, json
import nibabel as nib
import numpy as np
from PIL import Image

DATASET_JSON = {
    "channel_names": {"0": "FLAIR"},
    "labels": {"background": 0, "tumor": 1},
    "numTraining": 0,
    "file_ending": ".nii.gz",
    "name": "Dataset001_TCGALGG",
    "description": "TCGA-LGG 2D FLAIR slices — 110 lower-grade glioma patients (5 TCIA institutions)",
    "reference": "https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation",
    "licence": "CC0 1.0",
    "overwrite_image_reader_writer": "SimpleITKIO",
}

def slice_to_nifti(tif_path, is_mask=False):
    arr = np.array(Image.open(tif_path))
    if arr.ndim == 3:
        arr = arr[:, :, 0]
    arr = (arr > 0).astype(np.uint8) if is_mask else arr.astype(np.float32)
    return nib.Nifti1Image(arr[:, :, np.newaxis], affine=np.eye(4))

kaggle_3m = Path(LGG_INPUT) / "kaggle_3m"
output_dir = NNUNET_RAW / "Dataset001_TCGALGG"
images_out = output_dir / "imagesTr"
labels_out = output_dir / "labelsTr"
images_out.mkdir(parents=True, exist_ok=True)
labels_out.mkdir(parents=True, exist_ok=True)

patient_dirs = sorted(p for p in kaggle_3m.iterdir() if p.is_dir())
print(f"Found {len(patient_dirs)} patients")

case_ids = []
for patient_dir in patient_dirs:
    pid = patient_dir.name
    for slice_file in sorted(f for f in patient_dir.glob("*.tif") if not f.stem.endswith("_mask")):
        mask_file = slice_file.parent / f"{slice_file.stem}_mask.tif"
        if not mask_file.exists():
            continue
        m = re.search(r"_(\d+)$", slice_file.stem)
        slice_num = int(m.group(1)) if m else 0
        case_id = f"{pid}_{slice_num:03d}"
        nib.save(slice_to_nifti(slice_file), images_out / f"{case_id}_0000.nii.gz")
        nib.save(slice_to_nifti(mask_file, is_mask=True), labels_out / f"{case_id}.nii.gz")
        case_ids.append(case_id)

DATASET_JSON["numTraining"] = len(case_ids)
with open(output_dir / "dataset.json", "w") as f:
    json.dump(DATASET_JSON, f, indent=4)

print(f"Converted {len(case_ids)} slices → {output_dir}")

## 4. Dataset fingerprinting and preprocessing

nnU-Net analyzes the dataset and creates a training plan. Takes a few minutes on CPU.

In [ ]:
run("nnUNetv2_plan_and_preprocess -d 1 -c 2d --verify_dataset_integrity")

## 5. Verify GPU and nnU-Net

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

import nnunetv2
print("nnunetv2 version:", nnunetv2.__version__)

## 6. Train fold

Runs for 1000 epochs (nnU-Net default). If the notebook disconnects, re-run with `--c` appended to resume from the latest checkpoint.

In [ ]:
import time
start = time.time()

train_cmd = f"nnUNetv2_train Dataset001_TCGALGG 2d {FOLD} --npz"
print(f"Running: {train_cmd}")
run(train_cmd)

elapsed = (time.time() - start) / 3600
print(f"\nTraining complete. Elapsed: {elapsed:.1f} hours")

## 7. Validation results

In [ ]:
import json

summary_path = (
    NNUNET_RESULTS
    / "Dataset001_TCGALGG"
    / "nnUNetTrainer__nnUNetPlans__2d"
    / f"fold_{FOLD}"
    / "validation"
    / "summary.json"
)

if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    mean = summary.get("foreground_mean", summary.get("mean", {}))
    print(f"Fold {FOLD} validation results:")
    print(f"  Mean Dice : {mean.get('Dice', 'N/A')}")
    print(f"  Mean HD95 : {mean.get('HD95', 'N/A')} mm")
    print(f"  Mean IoU  : {mean.get('IoU', 'N/A')}")
    print(f"\nFull summary: {summary_path}")
else:
    print(f"summary.json not found at {summary_path}")
    print("Training may not have completed.")

## 8. Package results for download

Zips the model checkpoint and validation summary. Download from the Kaggle output panel.

In [ ]:
results_fold_dir = (
    NNUNET_RESULTS
    / "Dataset001_TCGALGG"
    / "nnUNetTrainer__nnUNetPlans__2d"
    / f"fold_{FOLD}"
)

zip_path = f"/kaggle/working/fold_{FOLD}_results.zip"
run(f"zip -r {zip_path} {results_fold_dir}")
print(f"Results saved to: {zip_path}")
print("Download from the Kaggle output panel on the right.")